In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,accuracy_score

In [2]:
DATA_DIR="/kaggle/input/datasets/mahmoudsalamagado/task-priority/Task_Prioritization_Dataset.csv"
df=pd.read_csv(DATA_DIR)
print("\nShape:",df.shape)
print("\nColumns:",list(df.columns))


Shape: (100, 10)

Columns: ['Task_ID', 'Task_Name', 'Priority', 'Deadline', 'Estimated_Time', 'Task_Type', 'Dependency', 'Employee_ID', 'Completion_Status', 'Urgency_Score']


In [3]:
print("\nSample rows:")
print(df.head())


Sample rows:
   Task_ID                        Task_Name Priority          Deadline  \
0        1  Task 1: Sample Task Description   Medium  2025-01-28 09:00   
1        2  Task 2: Sample Task Description      Low  2025-01-28 10:00   
2        3  Task 3: Sample Task Description     High  2025-01-28 11:00   
3        4  Task 4: Sample Task Description   Medium  2025-01-28 12:00   
4        5  Task 5: Sample Task Description      Low  2025-01-28 13:00   

  Estimated_Time Task_Type Dependency  Employee_ID Completion_Status  \
0        2 hours  Analysis        NaN          101       In Progress   
1        3 hours    Review  Task_ID 1          102         Completed   
2        4 hours   Meeting  Task_ID 2          103           Pending   
3        5 hours  Planning  Task_ID 3          104       In Progress   
4        1 hours  Analysis  Task_ID 4          105         Completed   

   Urgency_Score  
0              2  
1              3  
2              4  
3              5  
4            

In [4]:
print("\nData types:\n",df.dtypes)


Data types:
 Task_ID               int64
Task_Name            object
Priority             object
Deadline             object
Estimated_Time       object
Task_Type            object
Dependency           object
Employee_ID           int64
Completion_Status    object
Urgency_Score         int64
dtype: object


In [5]:
possible_targets=[c for c in df.columns if "priorit" in c.lower()]
target_col=possible_targets[0] if possible_targets else df.columns[-1]
print(f"\nUsing target column: '{target_col}'")


Using target column: 'Priority'


In [6]:
text_cols=[c for c in df.columns if df[c].dtype==object and c!=target_col]
numeric_cols=[c for c in df.columns if df[c].dtype!=object and c!=target_col]
print(f"Text/categorical columns: {text_cols}")
print(f"Numeric columns: {numeric_cols}")

Text/categorical columns: ['Task_Name', 'Deadline', 'Estimated_Time', 'Task_Type', 'Dependency', 'Completion_Status']
Numeric columns: ['Task_ID', 'Employee_ID', 'Urgency_Score']


In [7]:
df=df.dropna(subset=[target_col]).reset_index(drop=True)
y_raw=df[target_col]
if y_raw.dtype==object:
    priority_order={"low":0,"medium":1,"high":2,"critical":3}
    y_lower=y_raw.astype(str).str.lower().str.strip()
    if set(y_lower.unique()).issubset(priority_order.keys()):
        y=y_lower.map(priority_order)
    else:
        le=LabelEncoder()
        y=le.fit_transform(y_raw)
else:
    y=y_raw

In [8]:
desc_col=None
for c in text_cols:
    if df[c].astype(str).str.len().mean()>15:
        desc_col=c
        break
cat_cols=[c for c in text_cols if c!=desc_col]
X=df.drop(columns=[target_col])

In [9]:
transformers=[]
if numeric_cols:
    transformers.append(("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric_cols))
if cat_cols:
    transformers.append(("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat_cols))
if desc_col:
    transformers.append(("text",TfidfVectorizer(max_features=200,stop_words="english"),desc_col))

In [10]:
preprocessor=ColumnTransformer(transformers,remainder="drop")

In [11]:
model=Pipeline([("prep",preprocessor),("clf",RandomForestClassifier(n_estimators=300,max_depth=None,random_state=42))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y if len(np.unique(y))>1 else None)
model.fit(X_train,y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Task_ID', 'Employee_ID',
                                                   'Urgency_Score']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Deadline', 'Estimated_Time',
                                                   'Task_Type', 'Dependency',
                                                   'Completion_Status']),
                                                 ('text',
                                                  TfidfVectorizer(max_features=200,
                                                                  stop_words='english'),
                                                  'Task_Name')])),
                ('clf',
                 RandomForestClassifier(n_estimators=300, random_state=42))])

In [12]:
preds=model.predict(X_test)
print("\nAccuracy:",accuracy_score(y_test,preds))
print("\nClassification report:\n",classification_report(y_test,preds))


Accuracy: 1.0

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       1.00      1.00      1.00         7
           2       1.00      1.00      1.00         6

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [14]:
def rank_tasks(new_tasks_df):
    proba=model.predict_proba(new_tasks_df)
    class_labels=model.named_steps["clf"].classes_
    scores=(proba*class_labels).sum(axis=1)
    result=new_tasks_df.copy()
    result["priority_score"]=scores
    return result.sort_values("priority_score",ascending=False).reset_index(drop=True)

In [16]:
my_tasks = pd.DataFrame([
    {
        "Task_ID": 101,
        "Task_Name": "Submit tax documents",
        "Deadline": "2026-08-16",
        "Estimated_Time": 2,
        "Task_Type": "Admin",
        "Dependency": "None",
        "Employee_ID": 1,
        "Completion_Status": "Not Started",
        "Urgency_Score": 9
    },
    {
        "Task_ID": 102,
        "Task_Name": "Reply to non-urgent email",
        "Deadline": "2026-09-01",
        "Estimated_Time": 0.25,
        "Task_Type": "Communication",
        "Dependency": "None",
        "Employee_ID": 1,
        "Completion_Status": "Not Started",
        "Urgency_Score": 2
    },
    {
        "Task_ID": 103,
        "Task_Name": "Fix production bug",
        "Deadline": "2026-08-15",
        "Estimated_Time": 4,
        "Task_Type": "Engineering",
        "Dependency": "None",
        "Employee_ID": 2,
        "Completion_Status": "In Progress",
        "Urgency_Score": 10
    },
    {
        "Task_ID": 104,
        "Task_Name": "Plan next quarter's roadmap",
        "Deadline": "2026-10-01",
        "Estimated_Time": 8,
        "Task_Type": "Planning",
        "Dependency": "Budget approval",
        "Employee_ID": 3,
        "Completion_Status": "Not Started",
        "Urgency_Score": 3
    },
])

ranked = rank_tasks(my_tasks)
print(ranked[["Task_Name", "priority_score"]])

                     Task_Name  priority_score
0           Fix production bug        1.010000
1         Submit tax documents        0.980000
2    Reply to non-urgent email        0.953333
3  Plan next quarter's roadmap        0.953333
